# Model v3: Elle Etiketlenmiş Türkiye Hasar Örnekleri + Telefon Modeli

| | v2 (uygulamada) | v3 (bu notebook) |
|---|---|---|
| Model | YOLO11m, 800 px | **Aynı** (v2 ağırlıklarından devam eder) |
| Veri | RDD2022 + Türkiye'den 598 sağlam yol (yarı otomatik) | Aynı + **elle etiketlenmiş Türkiye adayları** (`hasar_etiket.zip`) |
| Test | RDD test bölümü + Bursa test seti | **Aynı** (karşılaştırma adil) |

v2'nin Bursa'daki asıl sorunu kaçırmaydı: eğitime Türkiye'den hiç hasarlı örnek girmemişti.
v3'te model ve çözünürlük değişmez, yalnızca veri eklenir; böylece iyileşmenin veriden geldiği ölçülebilir.
Etiket paketi iki tür örnek taşır:
- **Hasarlı:** kutuları elle düzeltilmiş Türk yolu hasarları
- **Hasarsız (zor örnek):** modelin hasar sandığı ama sağlam olan yollar (rögar, şerit, parke, gölge): yanlış alarmı azaltır

İsteğe bağlı ikinci eğitim: canlı tarama (`/canli`) için telefonda çalışacak **küçük model** (YOLO11n, 640 px).

## Çalıştırma
1. Drive'daki `yol-hasar-tespit` klasöründe şunlar olsun: `turkiye_egitim.zip`, `hasar_etiket.zip` (yerelde `python scripts/etiket_paketle.py` ile üretilir) ve `v2/best_v2.pt`.
2. `Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU`, sonra `Çalışma zamanı → Tümünü çalıştır`.
3. Drive iznini onaylayın. Oturum koparsa yeniden `Tümünü çalıştır`: eğitimler kaldığı yerden devam eder.

Sonuçlar Drive'da `yol-hasar-tespit/v3/` klasörüne yazılır.

## 1. Ortam

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU YOK! Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU, sonra baştan çalıştırın.'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/yol-hasar-tespit'
OUT = f'{DRIVE_DIR}/v3'
os.makedirs(OUT, exist_ok=True)

!git clone -q https://github.com/mahmutsamiseyhan/yol-hasar-tespit.git /content/yol-hasar-tespit || (cd /content/yol-hasar-tespit && git pull -q)
%cd /content/yol-hasar-tespit
!pip install -q ultralytics remotezip onnx onnxslim

## 2. Veri (v1 ve v2 ile aynı ülkeler ve aynı bölümler, seed=42)

In [ ]:
!python scripts/download_rdd2022.py --out /content/rdd_raw
!python scripts/prepare_rdd2022.py --src /content/rdd_raw --out /content/rdd_yolo
DATA_YAML = '/content/rdd_yolo/data.yaml'

## 2b. Ek veri (yalnızca eğitim bölümüne)

Elle etiketlenmiş Türkiye örnekleri 22 bin RDD görüntüsünün yanında çok az kalır; model onları daha sık görsün diye
eğitim bölümüne `TEKRAR` kez kopyalanır (her kopya eğitimde farklı kırpma/renk değişimiyle görülür).
Test ve doğrulama bölümlerine hiçbir şey eklenmez.

In [ ]:
import zipfile, shutil, json
from pathlib import Path
TEKRAR = 3
EK_PAKETLER = ([(Path(DRIVE_DIR) / 'turkiye_egitim.zip', 1)]
               + [(p, TEKRAR) for p in sorted(Path(DRIVE_DIR).glob('hasar_etiket*.zip'))]
               + [(p, TEKRAR) for p in sorted(Path(DRIVE_DIR).glob('saha_egitim*.zip'))])
assert any('hasar_etiket' in p.name for p, _ in EK_PAKETLER), "hasar_etiket.zip Drive'da yok: v3'ün asıl verisi bu."
for paket, tekrar in EK_PAKETLER:
    if not paket.exists():
        print(f"UYARI: {paket.name} Drive'da yok, atlandı.")
        continue
    hedef = Path('/content/ek') / paket.stem
    shutil.rmtree(hedef, ignore_errors=True)
    with zipfile.ZipFile(paket) as z:
        z.extractall(hedef)
    n = 0
    for img in (hedef / 'images').glob('*.jpg'):
        for k in range(tekrar):
            ad = img.stem if k == 0 else f'{img.stem}_k{k}'
            shutil.copy(img, f'/content/rdd_yolo/images/train/{ad}.jpg')
            shutil.copy(hedef / 'labels' / f'{img.stem}.txt', f'/content/rdd_yolo/labels/train/{ad}.txt')
        n += 1
    rapor = json.load(open(hedef / 'rapor.json'))
    print(f'{paket.name}: {n} görüntü × {tekrar} eğitime eklendi.', rapor.get('kararlar'), rapor.get('kutular'))
Path('/content/rdd_yolo/labels/train.cache').unlink(missing_ok=True)  # YOLO etiketleri yeniden tarasın

## 3. Eğitim (v3): v2 ağırlıklarından devam

Sıfırdan değil, v2'nin öğrendiklerinin üstüne eğitilir: daha kısa sürer (30 epoch) ve model/çözünürlük aynı kaldığı için
v2 ile fark yalnızca veriden gelir. 10 epoch iyileşme olmazsa erken durur.

In [ ]:
from ultralytics import YOLO

BEST_V2 = Path(DRIVE_DIR) / 'v2' / 'best_v2.pt'
assert BEST_V2.exists(), f'v2 modeli bulunamadı: {BEST_V2}'
IMGSZ, EPOCHS, BATCH = 800, 30, 16
RUN_NAME = 'rdd2022_yolo11m_800_v3'
PROJECT = f'{DRIVE_DIR}/runs'

def egit(baslangic, run_name, **kw):
    last = Path(PROJECT) / run_name / 'weights' / 'last.pt'
    if last.exists():
        try:
            print('Önceki eğitim bulundu, kaldığı yerden devam ediliyor:', last)
            m = YOLO(str(last))
            m.train(resume=True)
            return Path(m.trainer.save_dir)
        except AssertionError as e:  # eğitim zaten bitmişse Ultralytics devam etmeyi reddeder
            print('Eğitim zaten tamamlanmış, sonuçlar kullanılıyor.', e)
            return last.parent.parent
    m = YOLO(str(baslangic))
    m.train(data=DATA_YAML, project=PROJECT, name=run_name, exist_ok=True, cos_lr=True, close_mosaic=5,
            seed=42, plots=True, workers=8, **kw)
    return Path(m.trainer.save_dir)

RUN_DIR = egit(BEST_V2, RUN_NAME, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=10)
BEST_V3 = RUN_DIR / 'weights' / 'best.pt'
shutil.copy(BEST_V3, f'{OUT}/best_v3.pt')
print('Kaydedildi:', f'{OUT}/best_v3.pt')

## 3b. Telefon modeli (canlı tarama, isteğe bağlı)

Telefonun tarayıcısı her kareyi tarar; YOLO11n (2,6 milyon parametre) YOLO11s'den yaklaşık 3 kat hızlıdır.
Telefonun gönderdiği kareye sunucu yine büyük modelle (v3) bakar. Atlamak için `EGIT_CANLI = False`.

In [ ]:
EGIT_CANLI = True
if EGIT_CANLI:
    CANLI_DIR = egit('yolo11n.pt', 'rdd2022_yolo11n_640_canli', epochs=80, imgsz=640, batch=32, patience=20)
    BEST_CANLI = CANLI_DIR / 'weights' / 'best.pt'
    onnx = YOLO(str(BEST_CANLI)).export(format='onnx', imgsz=640, simplify=True, dynamic=False, opset=17)
    shutil.copy(BEST_CANLI, f'{OUT}/best_canli.pt')
    shutil.copy(onnx, f'{OUT}/canli.onnx')
    print('Kaydedildi:', f'{OUT}/canli.onnx')

## 4. Değerlendirme: aynı test setinde v2, v3 (ve telefon modeli)

- **mAP50 / AP50:** standart nesne tespiti başarı notu (eşikten bağımsız)
- **Sınıf bazlı eşikler:** doğrulama setinde F1'i en yüksek yapan güven eşiği; uygulama bu eşiklerle karar verir.

Asıl karar Bursa test setinde verilir (yerelde `scripts/evaluate_bursa.py`); RDD sonucu modelin genel bilgisini
kaybetmediğini gösterir.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display

MODELS = {'v2 (YOLO11m, 800)': (str(BEST_V2), 800), 'v3 (YOLO11m, 800)': (str(BEST_V3), IMGSZ)}
if EGIT_CANLI:
    MODELS['canli (YOLO11n, 640)'] = (str(BEST_CANLI), 640)

def f1_thresholds(metrics, names, lo=0.05, hi=0.9, fallback=0.35):
    # Sınıf başına F1'i en yüksek yapan eşik; F1 çok düşükse (sınıf öğrenilememiş) genel eşik kullanılır
    b = metrics.box
    out = {}
    for i, cls_id in enumerate(b.ap_class_index):
        f1 = np.asarray(b.f1_curve)[i]
        j = int(f1.argmax())
        esik = float(np.asarray(b.px)[j]) if f1[j] >= 0.1 else fallback
        out[names[int(cls_id)]] = {'esik': round(min(hi, max(lo, esik)), 3), 'f1': round(float(f1[j]), 3)}
    return out

results, rows = {}, []
for name, (path, sz) in MODELS.items():
    m = YOLO(path)
    val = m.val(data=DATA_YAML, split='val', imgsz=sz, batch=16, plots=False, verbose=False)
    test = m.val(data=DATA_YAML, split='test', imgsz=sz, batch=16, plots=True, verbose=False,
                 project=OUT, name=name.split()[0] + '_test', exist_ok=True)
    th = f1_thresholds(val, m.names)
    per_class = {}
    for i, cls_id in enumerate(test.box.ap_class_index):
        p, r, ap50, ap = test.box.class_result(i)
        per_class[m.names[int(cls_id)]] = round(float(ap50), 3)
    results[name] = {'mAP50': round(float(test.box.map50), 3), 'mAP50-95': round(float(test.box.map), 3),
                     'precision': round(float(test.box.mp), 3), 'recall': round(float(test.box.mr), 3),
                     'AP50': per_class, 'esikler': th, 'hiz_ms': round(test.speed['inference'], 1)}
    rows.append({'model': name, **{k: results[name][k] for k in ('mAP50', 'mAP50-95', 'precision', 'recall')},
                 **{f'AP50 {c}': v for c, v in per_class.items()}, 'çıkarım ms': results[name]['hiz_ms']})
    tag = name.split()[0]
    json.dump({'esikler': {c: v['esik'] for c, v in th.items()},
               'kaynak': f'RDD2022 doğrulama seti, sınıf başına en yüksek F1 ({name})'},
              open(f'{OUT}/esikler_{tag}.json', 'w'), indent=2, ensure_ascii=False)

display(pd.DataFrame(rows).set_index('model'))
json.dump(results, open(f'{OUT}/kiyas_v2_v3.json', 'w'), indent=2, ensure_ascii=False)
print('\nDrive:', OUT)
!ls -la {OUT}

## 5. Sonra yapılacaklar
Drive'daki `yol-hasar-tespit/v3/` klasöründen şunları indirip `C:\Projeler\yol-hasar-tespit\models\` içine koyun:
- `best_v3.pt`, `esikler_v3.json`, `kiyas_v2_v3.json`
- telefon modeli eğitildiyse `canli.onnx` ve `esikler_canli.json`

Sonra Claude'a "v3 dosyalarını koydum" yazın. v2 ile v3 Bursa test setinde ölçülür; daha iyi olan uygulamaya alınır.